# Group Comparison — t-Test / Mann-Whitney U

Compares a metric between two groups. Automatically selects parametric (t-test) or
non-parametric (Mann-Whitney U) test based on a Shapiro-Wilk normality check.

**Usage:**
1. Load the relevant DataFrame in Section 1.
2. Build derived columns and join grouping variables in Section 2.
3. Set `group_col` and `metric_col` in Section 3 and run.

In [ ]:
# Section 1 — Setup & Data Loading
from utils.db import load_handover, load_participants
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt

%matplotlib inline

# Load data — add study_id=N or experiment_id=N to filter
df = load_handover()

In [ ]:
# Section 2 — Data Preparation

# Convert timestamp columns
for col in ['giver_grasped_object', 'giver_released_object']:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# Derive handover duration
df['duration_ms'] = (
    df['giver_released_object'] - df['giver_grasped_object']
).dt.total_seconds() * 1000

# --- Add joins and derived grouping columns here ---
# Example: join participants to group by age range
# df_p = load_participants()
# df = df.merge(df_p[['participant_id', 'age']], left_on='giver', right_on='participant_id')
# df['age_group'] = pd.cut(df['age'], bins=[0, 30, 100], labels=['young', 'older'])

print(df[['duration_ms', 'is_error']].describe())

In [ ]:
# Section 3 — Analysis
# Configure the comparison here:
group_col  = 'is_error'      # column with exactly 2 unique values (must exist after prep)
metric_col = 'duration_ms'   # numeric metric to compare

groups = df[group_col].dropna().unique()
assert len(groups) == 2, f'Expected 2 groups, found: {groups}'

group_a = df[df[group_col] == groups[0]][metric_col].dropna()
group_b = df[df[group_col] == groups[1]][metric_col].dropna()

print(f'Group {groups[0]}: n={len(group_a)}, mean={group_a.mean():.2f}')
print(f'Group {groups[1]}: n={len(group_b)}, mean={group_b.mean():.2f}')

In [ ]:
# Normality check (Shapiro-Wilk; reliable for n < 50)
# Note: checks only group_a — for rigorous analysis check both groups
_, p_norm = stats.shapiro(group_a)
print(f'Shapiro-Wilk p={p_norm:.4f} ({"normal" if p_norm > 0.05 else "non-normal"})')

if p_norm > 0.05:
    stat, p = stats.ttest_ind(group_a, group_b)
    test_name = 'Independent t-test'
else:
    stat, p = stats.mannwhitneyu(group_a, group_b, alternative='two-sided')
    test_name = 'Mann-Whitney U'

print(f'\n{test_name}: stat={stat:.3f}, p={p:.4f}')
print('Significant (p < 0.05):', p < 0.05)

In [ ]:
# Section 4 — Visualisation
df.boxplot(column=metric_col, by=group_col)
plt.title(f'{metric_col} by {group_col}')
plt.suptitle('')
plt.tight_layout()
plt.show()